**<span style="color:#809bd8">Curso:</span>**
**Infraestructura y Aqrquitectura Para Big Data**

**<span style="color:#809bd8">Tema:</span>**
**Evidencia de Aprendizaje EA1**

**<span style="color:#809bd8">Grupo:</span>**
**PREICA2601B010094**

**<span style="color:#809bd8">Estudiante:</span>**
**Eduard Andres Paez Bohorquez**

**<span style="color:#809bd8">Profesor:</span>**
**Walter Hugo Arboleda Mazo**

**<span style="color:#809bd8">Universidad:</span>**
**Institución Universitaria Digital de Antioquia - IUDigital**

**<span style="color:#809bd8">Fecha:</span>**
**03/03/2026**

In [3]:
# ============================
# INSTALAR DEPENDENCIAS
# ============================
! pip install requests pandas openpyxl

In [4]:
# ============================
# IMPORTAR LIBRERÍAS
# ============================
import requests # librería para hacer solicitudes HTTP
import sqlite3 # librería para trabajar con bases de datos SQLite
import pandas as pd # librería para manipulación de datos
import os # librería para trabajar con el sistema de archivos
from datetime import datetime # librería para trabajar con fechas y horas

# API de tipo REST para obtener tasas de cambio de divisas
API_URL = "https://open.er-api.com/v6/latest/USD"

# Configuración de rutas para almacenar archivos
DB_DIR = "datos_api" # ruta para la base de datos SQLite
CSV_DIR = "exchange_rates" # ruta para el archivo CSV
AUDIT_DIR = "static/auditoria" # ruta para el archivo de auditoría

for folder in [DB_DIR, CSV_DIR, AUDIT_DIR]:
        os.makedirs(folder, exist_ok=True) # crear las carpetas si no existen

# creación de la funcion extraer_datos() para obtener los datos de la API
def extraer_datos(url):
    response = requests.get(url) # hacer una solicitud GET a la API
    if response.status_code == 200: # verificar si la solicitud fue exitosa
        data = response.json() # convertir la respuesta a formato JSON
        return data # retornar los datos obtenidos
    else:
        raise Exception(f"Error al obtener datos: {response.status_code}") # lanzar una excepción si la solicitud no fue exitosa
    
# Almacenamiento de datos en una base de datos SQLite
def almacenar_datos(data):
    db_path = os.path.join(DB_DIR, 'ingestion.db') # construir la ruta completa para la base de datos
    conn = sqlite3.connect(db_path) # conectar a la base de datos SQLite
    cursor = conn.cursor() # crear un cursor para ejecutar comandos SQL
    
    # Crear la tabla si no existe
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS exchange_rates (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            currency TEXT NOT NULL,
            rate REAL NOT NULL,
            timestamp TEXT NOT NULL
        )
    ''')
    
    # Insertar los datos en la tabla
    for currency, rate in data['rates'].items():
        cursor.execute('''
            INSERT INTO exchange_rates (currency, rate, timestamp)
            VALUES (?, ?, ?)
        ''', (currency, rate, datetime.now().isoformat()))
    
    conn.commit() # guardar los cambios en la base de datos
    conn.close() # cerrar la conexión a la base de datos
    return db_path # retornar la ruta de la base de datos creada

# Análisis de datos utilizando pandas
def analizar_y_auditar(datos_api):
    db_path = os.path.join(DB_DIR, 'ingestion.db') # construir la ruta completa para la base de datos
    conn = sqlite3.connect(db_path) # conectar a la base de datos SQLite

    # Leer los datos de la base de datos en un DataFrame de pandas
    df = pd.read_sql_query("SELECT * FROM exchange_rates", conn)
    csv_path = os.path.join(CSV_DIR, 'exchange_rates.csv') # construir la ruta completa para el archivo CSV
    df.to_csv(csv_path, index=False) # guardar el DataFrame en un archivo CSV

    # Logica de auditoría para comparar los datos obtenidos de la API con los datos almacenados en la base de datos
    # Usamos la función len() para contar el número de registros en ambos conjuntos de datos y luego escribimos un resumen de la auditoría en un archivo de texto.
    registos_api = len(datos_api['rates']) # contar el número de registros obtenidos de la API
    registros_db = len(df) # contar el número de registros almacenados en la base de

    audi_path = os.path.join(AUDIT_DIR, 'auditoria.txt') # construir la ruta completa para el archivo de auditoría
    with open(audi_path, 'w', encoding='utf-8') as f:
        f.write(f"RESUMEN DE LA AUDITORÍA:\n")
        f.write(f"====================\n")
        f.write(f"Registros obtenidos desde la API: {registos_api}\n")
        f.write(f"Registros almacenados en la base de datos: {registros_db}\n")
        f.write(f"Fecha y hora de auditoría: {datetime.now().isoformat()}\n")

        if registos_api >= registros_db: # >= porque si corre el proceso varias veces, la base de datos puede tener más registros que la API debido a inserciones repetidas
            f.write("Auditoría exitosa: Todos los registros de la API fueron almacenados correctamente en la base de datos.\n")
        else:
            f.write("Auditoría fallida: Hay una discrepancia entre los registros obtenidos de la API y los almacenados en la base de datos.\n") 

    conn.close() # cerrar la conexión a la base de datos

# Función principal para ejecutar el proceso completo
if __name__ == "__main__":
    try:
        print("Iniciando proceso de Big Data EA1...")

        datos = extraer_datos(API_URL)
        print(f"Datos extraídos (Moneda base: {datos['base_code']})")

        almacenar_datos(datos)
        print(f"Datos guardados en {DB_DIR}/ingestion.db")

        analizar_y_auditar(datos)
        print(f"Archivos de evidencia generados en {CSV_DIR} y {AUDIT_DIR}")

        print("\n¡Todo listo para subir a GitHub!")
    except Exception as e:
        print(f"Ocurrió un error: {e}")
        

Iniciando proceso de Big Data EA1...
Datos extraídos (Moneda base: USD)
Datos guardados en datos_api/ingestion.db
Archivos de evidencia generados en exchange_rates y static/auditoria

¡Todo listo para subir a GitHub!


### Referencias Bibliográficas

* **Apache Software Foundation.** (2023). *SQLite Documentation*. https://www.sqlite.org/docs.html
* **Community, T. P. D.** (2024). *pandas: Powerful data analysis toolkit*. PyPI. https://pandas.pydata.org/docs/
* **GitHub, Inc.** (2024). *GitHub Actions Documentation: Understanding GitHub Actions*. https://docs.github.com/en/actions
* **Open-Meteo & Exchange Rates API.** (2024). *Standard V6 Currency API Documentation*. https://www.exchangerate-api.com/docs/free
* **Python Software Foundation.** (2024). *Python Language Reference, version 3.10*. https://www.python.org/
* **Reitz, K.** (2023). *Requests: HTTP for Humans*. https://requests.readthedocs.io/en/latest/
* **The Jupyter Team.** (2024). *Jupyter Notebook Documentation*. https://jupyter-notebook.readthedocs.io/